# AI Salary Prediction Notebook

This notebook trains a salary model with the approved business input columns, saves the selected model locally, and exposes a Gradio app backed by a second inference pipeline.


## Pipeline Layout

1. Training pipeline: preprocess -> train both models -> evaluate -> select best -> pickle artifact.
2. Inference pipeline: load pickled model -> run inference from Gradio input -> generate business interpretation with the LLM.


In [11]:
!pip install -q langgraph openai pandas scikit-learn pydantic numpy kagglehub python-dotenv xgboost gradio matplotlib shap

import json
import os
import pickle
from operator import add
from typing import Annotated, Any, Optional

import gradio as gr
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from langgraph.graph import END, StateGraph
from openai import OpenAI
from pydantic import BaseModel, Field
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# remove the 3 lines below and uncomment the 4th line if you want to run this code in Google Colab
from dotenv import load_dotenv
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

OPENAI_MODEL = "gpt-4o-mini"
client = OpenAI()

MODEL_PATH = "models/selected_model.pkl"
TARGET_COLUMN = "salary_usd"
TRAINING_COLUMNS = [
    "job_role",
    "experience_level",
    "experience_years",
    "country",
    "education_required",
    "industry",
    "company_size",
    "work_mode",
]
CATEGORICAL_TRAINING_COLUMNS = [
    "job_role",
    "experience_level",
    "country",
    "education_required",
    "industry",
    "company_size",
    "work_mode",
]
NUMERIC_TRAINING_COLUMNS = ["experience_years"]

os.makedirs("models", exist_ok=True)
for skill_dir in [
    "skills/preprocess",
    "skills/random_forest_train",
    "skills/xgboost_train",
    "skills/random_forest_eval",
    "skills/xgboost_eval",
    "skills/model_selection",
    "skills/load_pickled_model",
    "skills/run_inference",
    "skills/generate_recommendation",
]:
    os.makedirs(skill_dir, exist_ok=True)

print("Environment ready.")


Environment ready.


In [12]:
path = kagglehub.dataset_download("mohankrishnathalla/global-ai-and-data-jobs-salary-dataset")
filename = os.listdir(path)[0]
job_df = pd.read_csv(os.path.join(path, filename))
job_df = job_df[TRAINING_COLUMNS + [TARGET_COLUMN]].copy()

FEATURE_CHOICES = {
    column: sorted(job_df[column].dropna().astype(str).unique().tolist())
    for column in CATEGORICAL_TRAINING_COLUMNS
}
EXPERIENCE_MIN = int(job_df["experience_years"].min())
EXPERIENCE_MAX = int(job_df["experience_years"].max())

print(f"Loaded {filename} with {len(job_df):,} rows.")
print("Training columns:", TRAINING_COLUMNS)
job_df.head()


Loaded global_ai_jobs.csv with 90,000 rows.
Training columns: ['job_role', 'experience_level', 'experience_years', 'country', 'education_required', 'industry', 'company_size', 'work_mode']


,job_role,experience_level,experience_years,country,education_required,industry,company_size,work_mode,salary_usd
0,Machine Learning Engineer,Entry,0,UAE,Master,Automotive,Small,Remote,66465
1,AI Engineer,Entry,1,USA,Bootcamp,Retail,Small,Onsite,75507
2,Research Scientist,Entry,0,Brazil,PhD,Healthcare,Large,Remote,41660
3,Software Engineer AI,Senior,6,India,Diploma,Tech,Large,Onsite,43268
4,Machine Learning Engineer,Entry,0,Germany,Master,Retail,Medium,Hybrid,69119


In [13]:
class ModelAgentState(BaseModel):
    messages: Annotated[list[str], add] = Field(default_factory=list)
    raw_df: Optional[Any] = None
    processed_df: Optional[Any] = None
    random_forest_model: Optional[Any] = None
    xgboost_model: Optional[Any] = None
    X_train: Optional[Any] = None
    y_train: Optional[Any] = None
    X_test: Optional[Any] = None
    y_test: Optional[Any] = None
    random_forest_evaluation_metrics: Optional[dict[str, float]] = None
    xgboost_evaluation_metrics: Optional[dict[str, float]] = None
    feature_columns: Optional[list[str]] = None
    selected_model: Optional[Any] = None
    selected_model_name: Optional[str] = None
    selection_reason: Optional[str] = None

    class Config:
        arbitrary_types_allowed = True


class PredictionAgentState(BaseModel):
    messages: Annotated[list[str], add] = Field(default_factory=list)
    raw_input: dict[str, Any] = Field(default_factory=dict)
    normalized_input: Optional[dict[str, Any]] = None
    model_artifact: Optional[dict[str, Any]] = None
    prepared_features: Optional[Any] = None
    prediction: Optional[float] = None
    prediction_low: Optional[float] = None
    prediction_high: Optional[float] = None
    shap_base_value: Optional[float] = None
    shap_top_factors: Optional[list[dict[str, Any]]] = None
    shap_summary: Optional[str] = None
    business_interpretation: Optional[str] = None

    class Config:
        arbitrary_types_allowed = True


def model_field_names(model_cls: type[BaseModel]) -> list[str]:
    if hasattr(model_cls, "model_fields"):
        return list(model_cls.model_fields.keys())
    return list(model_cls.__fields__.keys())


print("ModelAgentState fields:", model_field_names(ModelAgentState))
print("PredictionAgentState fields:", model_field_names(PredictionAgentState))


ModelAgentState fields: ['messages', 'raw_df', 'processed_df', 'random_forest_model', 'xgboost_model', 'X_train', 'y_train', 'X_test', 'y_test', 'random_forest_evaluation_metrics', 'xgboost_evaluation_metrics', 'feature_columns', 'selected_model', 'selected_model_name', 'selection_reason']
PredictionAgentState fields: ['messages', 'raw_input', 'normalized_input', 'model_artifact', 'prepared_features', 'prediction', 'prediction_low', 'prediction_high', 'shap_base_value', 'shap_top_factors', 'shap_summary', 'business_interpretation']


/tmp/ipykernel_4075013/220539760.py:1: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class ModelAgentState(BaseModel):
/tmp/ipykernel_4075013/220539760.py:22: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class PredictionAgentState(BaseModel):


In [14]:
SKILL_SPECS = {
    'preprocess': {
        "path": 'skills/preprocess/SKILL.md',
        "text": "---\nname: preprocess\ndescription: Prepare the AI & Data Jobs salary dataset for model training with deterministic cleaning, one-hot encoding, and a fixed train/test split.\ntags: [preprocessing, data ingestion, train-test split, encoding]\nmode: organisational\n---\n\n# Preprocess Skill\n\n## When to use\nInvoke this skill immediately after ingestion_agent has loaded the raw dataframe into state.raw_df.\n\n## Specification\n1. Read the raw dataframe from state.raw_df.\n2. Separate salary_usd as the regression target.\n3. Drop identifier columns that should not be used for training.\n4. Fill missing categorical values with 'Unknown' and numeric values with the median.\n5. One-hot encode categorical columns using pandas.\n6. Create a deterministic train/test split with random_state=42.\n7. Store processed_df, X_train, X_test, y_train, y_test, and feature_columns back into ModelAgentState.\n",
    },
    'random_forest_train': {
        "path": 'skills/random_forest_train/SKILL.md',
        "text": '---\nname: random_forest_train\ndescription: Train a Random Forest Regressor on the preprocessed AI & Data Jobs salary dataset using a fixed train/test split. Deterministic Python, no LLM involved.\ntags: [training, random forest, regression, sklearn]\nmode: organisational\n---\n\n# Model 1 (Random Forest) Training Skill\n\n## Role\nThis is an organisational skill. Model fitting with fixed hyperparameters is deterministic. The LLM is not involved. The Python code in random_forest_train_agent was written to satisfy this specification.\n\n## When to use\nInvoke this skill immediately after preprocess_agent has written X_train, y_train, and feature_columns to agent state. This skill must run before random_forest_eval_agent.\nIt runs in parallel with xgboost_train_agent and has no dependency on it.\n\n## Specification\n1. Read X_train, y_train, and feature_columns from state.\n2. Instantiate RandomForestRegressor(n_estimators=200, min_samples_leaf=4,\n   random_state=42, n_jobs=-1).\n3. Fit the model on X_train and y_train.\n4. Store the fitted model in state.random_forest_model.\n\n## Inputs\n- X_train: pd.DataFrame, one-hot encoded training features from preprocess_agent\n- y_train: pd.Series, continuous salary target values for the training split\n- feature_columns: list[str], ordered list of feature names after encoding\n\n## Outputs\n- random_forest_model: fitted RandomForestRegressor object\n\n## Output format\nrandom_forest_model is a scikit-learn fitted estimator object, ready to be consumed directly by random_forest_eval_agent via state. The field name itself identifies the model type for downstream agents and the\nGradio interface.\n\n## Business context\nThis stage produces the first candidate model that will ultimately predict salary ranges for AI and data roles. The output feeds directly into the evaluation and selection stages, which together determine which model\na hiring manager or job seeker sees results from in the final interface.\n\n## Critical Rule\nDo not perform any feature engineering or encoding here. All transformations must have been completed by preprocess_agent before this skill runs.\n',
    },
    'xgboost_train': {
        "path": 'skills/xgboost_train/SKILL.md',
        "text": '---\nname: xgboost_train\ndescription: Train an XGBoost Regressor on the preprocessed AI & Data Jobs salary dataset using a fixed train/test split. Deterministic Python, no LLM involved.\ntags: [training, xgboost, regression, gradient boosting]\nmode: organisational\n---\n\n# Model 2 (XGBoost) Training Skill\n\n## Role\nThis is an organisational skill. Model fitting with fixed hyperparameters is deterministic. The LLM is not involved. The Python code in xgboost_train_agent was written to satisfy this specification.\n\n## When to use\nInvoke this skill immediately after preprocess_agent has written X_train, y_train, and feature_columns to agent state. This skill must run before xgboost_eval_agent.\nIt runs in parallel with random_forest_train_agent and has no dependency on it.\n\n## Specification\n1. Read X_train, y_train, and feature_columns from state.\n2. Instantiate XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,\n   subsample=0.8, colsample_bytree=0.8, random_state=42, tree_method="hist").\n3. Fit the model on X_train and y_train.\n4. Store the fitted model in state.xgboost_model.\n\n## Inputs\n- X_train: pd.DataFrame, one-hot encoded training features from preprocess_agent\n- y_train: pd.Series, continuous salary target values for the training split\n- feature_columns: list[str], ordered list of feature names after encoding\n\n## Outputs\n- xgboost_model: fitted XGBRegressor object\n\n## Output format\nxgboost_model is an XGBoost fitted estimator object, ready to be consumed directly by xgboost_eval_agent via state. XGBoost accepts pd.DataFrame inputs natively; no numpy conversion is needed before passing X_train.\n\n## Business context\nThis stage produces the second candidate model for predicting salary ranges for AI and data roles. Gradient boosting is included as a comparison against Random Forest because it often captures complex non-linear feature\ninteractions in structured tabular data, which is relevant here given the mix of categorical job attributes driving salary variation.\n\n## Critical Rule\nDo not perform any feature engineering or encoding here. All transformations must have been completed by preprocess_agent before this skill runs.\n',
    },
    'random_forest_eval': {
        "path": 'skills/random_forest_eval/SKILL.md',
        "text": '---\n\nname: random_forest_evaluation_agent\ndescription: Evaluate the Random Forest model performance using regression metrics (RMSE, MAE, and R²) and visualizations.\nmode: LLM-assisted\ntags: [evaluation, random forest, regression\n---\n\n## When to use\nExecute after random_forest_train_agent has completed training.\n\n## How to execute\n1. Load trained model from state.random_forest_model\n2. Predict on the held-out test set (X_test)\n3. Calculate the metrics:\n   - RMSE: measure the average deviation between predicted and actual values.\n   - MAE: measure the average absolute difference between predicted and actual values.\n   - R²: measure the proportion of variance in the target variable that is explained by the model.\n4. Generate scatter plot (predicted vs actual)\n5. Generate residual plot showing errors acrocss salary range\n6. Use GPT-4o-mini to interpret metrics in plain English for non-technical stakeholders (HR)\n7. Store the evaluation metrics in state.random_forest_evaluation_metrics\n\n## Inputs from agent state\n- random_forest_model: Trained model\n- X_test: Test features\n- y_test: True test salaries\n\n## Outputs to agent state\n- random_forest_evaluation_metrics: dict with \'rmse\', \'mae\', \'r2\'\n- Messages: Plain English interpretation\n\n## Output format\nMetrics example:\n{\n    \'rmse\': 15234.56,\n    \'mae\': 12045.23,\n    \'r2\': 0.7623\n}\n\n## Notes\n- Use the TEST set only, do not evaluate the train data\n- GPT-4o-mini promot: "Explain what RMSE, MAE, and R² mean in the context of salary prediction accuracy. Keep in mind that the audiance is an HR"\n',
    },
    'xgboost_eval': {
        "path": 'skills/xgboost_eval/SKILL.md',
        "text": "---\n\nname: xgboost_evaluation_model\ndescription: Evaluate XGBoost model using RMSE, MAE, R² and visualizations\n---\n\n# XGBoost Evaluation Skill\n\n## When to use\nExecute after xgboost_train_agent has completed training.\n\n## How to execute\nIt is the similar process to the random forest evaluation, but for XGBoost.\n\n1. Load trained model from state.xgboost_model\n2. Predict on the hold-out test set X_test (NOT the training set)\n3. Calculate RMSE, MAE, R²\n4. Generate scatter plot (predicted vs actual)\n5. Generate residual plot\n6. Use GPT-4o-mini to interpret metrics in plain English that HR can easily undertand\n7. Store the evaluation metrics in state.xgboost_evaluation_metrics\n\n## Inputs from agent state\n- xgboost_model: Trained model\n- X_test: Test features\n- y_test: True test salaries\n\n## Outputs to agent state\n- xgboost_evaluation_metrics: dictionary with 'rmse', 'mae', 'r2'\n- Messages: Plain English interpretation from GPT\n- Update state message with evaluation summary\n\n## Output format\nMetrics example:\n{\n    'rmse': 1234.56,\n    'mae': 1234.56,\n    'r2': 0.1234\n}\n",
    },
    'model_selection': {
        "path": 'skills/model_selection/SKILL.md',
        "text": '---\nname: model_selection\ndescription: Compare the results of two models and select the better one and give the reason\nmode: LLM-driven\n---\n\n# Model Selection Skill\n\n## When to use\n- User asks to compare two models\n- User wonders which model performs better in prediction\n- User wants to choose a better model from candidate models\n\n## How to execute\n- Receive the evaluation metrics (RMSE, MAE, and R2) of the two models from ModelAgentState\n- Compare the metrics and select the model with a lower RMSE, a lower MAE, and a higher R2\n- If the three metrics have conflicts, choose the model with a lower RMSE\n- Store the selected model and reason as selected_model and selection_reason in ModelAgentState\n- Tell the user which model is better and provide its evaluation metrics\n\n## Input from agent state\n- random_forest_evaluation_metrics: dict\n- xgboost_evaluation_metrics: dict\n\n## Output to agent state\nStore the selected model and reason as selected_model and selection_reason in ModelAgentState\n\n## Output format\nAlways return output in json format {"selected_model": selected_model, "model_name": model_name, "selection_reason": reasoning}\n- selected_model is either random_forest_model or xgboost_model\n- model_name is either "Random Forest" or "XGBoost"\n- reasoning is a string starting "[model_name] performs better because ...", followed by the comparison of evaluation metrics, such as a lower RMSE, or a higher R2\n\n## Constraints\n- reasoning should not exceed 150 words total\n- Use plain, accessible language\n- Get the evaluation metrics from evaluation agents. Do not use fake numbers\n',
    },
    'run_inference': {
        "path": 'skills/run_inference/SKILL.md',
        "text": '---\nname: run_inference\ndescription: Predict the salary for a given job title, experience level and educational background, using the chosen model by model selection agent.\nUse when the user inputs the job requirements and expects a estimated salary\nmode: LLM-driven\n---\n\n# Run Inference Skill\n\n## When to use\n- User wants an estimated salary for a given job title, experience level and educational background\n- User requests a recommended salary for a data or AI related job posting\n- User wants to know the market rate for hiring a person in a given position\n\n## How to execute\n- LLM reads user prompt which contains key features\n- Get selected_model from ModelAgentState\n- Use the given features to predict the salary using the selected_model\n\n## Input from agent state\n- Store the key features into PredictionAgentState\n- Get selected_model from models/selected_model.pkl\n\n## Output to agent state\n- Store the predicted salary as prediction in PredictionAgentState\n\n## Output format\nAlways structure your response like this:\n- Respond with "Recommended salary for [job title] with [experience level] years of experience and [educational background] is $[salary range]."\n- The salary range is calculated by the model prediction with a 10% upper and lower margin\n\n## Constraints\n- Use plain, accessible language\n- Predicted salary should be from the chosen model\n- Never provide fake numbers\n',
    },
    'load_pickled_model': {
        "path": 'skills/load_pickled_model/SKILL.md',
        "text": '---\nname: load_pickled_model\ndescription: Load the locally saved selected model artifact from disk for inference.\ntags: [inference, model loading, pickle]\nmode: organisational\n---\n\n# Load Pickled Model Skill\n\n## Role\nThis is an organisational skill. The operation is deterministic and does not require the LLM. The Python code in `load_pickled_model_agent` is responsible for executing this specification exactly.\n\n## When to use\nInvoke this skill at the beginning of the prediction pipeline before any ETL, feature alignment, or model inference occurs.\n\n## Specification\n1. Open `models/selected_model.pkl` from local storage.\n2. Validate that the artifact exists and can be unpickled successfully.\n3. Confirm the artifact contains the fitted model object, selected model name, feature columns, and training metadata required by downstream agents.\n4. Store the loaded artifact in `PredictionAgentState.model_artifact`.\n\n## Inputs\n- Local file path: `models/selected_model.pkl`\n\n## Outputs\n- `model_artifact`: dictionary containing the trained estimator and its metadata\n\n## Output format\nThe loaded object should remain in its original Python structure so downstream agents can access:\n- `model`\n- `model_name`\n- `selection_reason`\n- `feature_columns`\n- `training_columns`\n- `categorical_columns`\n- `numeric_columns`\n\n## Business context\nThis stage ensures the frontend always uses the latest approved trained model selected by the training pipeline, rather than retraining on demand or relying on hard-coded objects in memory.\n\n## Critical Rule\nIf the artifact is missing or malformed, fail fast with a clear error instead of silently continuing with a fallback model.\n',
    },
    'generate_recommendation': {
        "path": 'skills/generate_recommendation/SKILL.md',
        "text": '---\nname: generate_recommendation\ndescription: Use the LLM configured through the local environment to turn the model prediction into a short business interpretation.\ntags: [inference, llm, business interpretation]\nmode: LLM-driven\n---\n\n# Generate Recommendation Skill\n\n## When to use\nExecute after the model has produced a salary estimate and the pipeline has a normalized user profile ready for presentation in the frontend.\n\n## How to execute\n1. Read the normalized job profile from `PredictionAgentState`.\n2. Read the predicted salary and suggested range from `PredictionAgentState`.\n3. Use the OpenAI API key configured in `.env` to call the LLM.\n4. Generate a concise business interpretation aimed at a hiring manager or compensation stakeholder.\n5. Keep the interpretation grounded in the model output and the submitted role context.\n\n## Inputs from agent state\n- `normalized_input`: cleaned frontend inputs after ETL\n- `prediction`: point estimate from the saved model\n- `prediction_low`: lower bound of the suggested range\n- `prediction_high`: upper bound of the suggested range\n- `model_artifact`: selected model metadata\n\n## Output to agent state\n- `business_interpretation`: markdown-ready explanation for the frontend\n\n## Output format\nThe response should be short, stakeholder-friendly, and suitable for direct rendering in Gradio. A compact bullet list is preferred.\n\n## Constraints\n- Do not invent a different salary number.\n- Do not contradict the predicted range.\n- Use plain business language, not ML jargon.\n- Keep the response concise enough for a frontend summary block.\n\n## Business context\nThis stage translates the raw numerical forecast into an explanation that helps users understand how to use the estimate in budgeting, benchmarking, and compensation discussions.\n',
    },
}

def load_skill_text(skill_name: str) -> str:
    return SKILL_SPECS[skill_name]["text"].strip()

def load_skill_body(skill_name: str) -> str:
    content = load_skill_text(skill_name)
    parts = content.split("---", maxsplit=2)
    if len(parts) < 3:
        return content
    return parts[2].strip()

def regenerate_skill_files() -> None:
    for spec in SKILL_SPECS.values():
        skill_path = spec["path"]
        os.makedirs(os.path.dirname(skill_path), exist_ok=True)
        with open(skill_path, "w", encoding="utf-8") as handle:
            handle.write(spec["text"].strip() + "\n")

regenerate_skill_files()
print("Skill files regenerated from in-notebook specs.")


Skill files regenerated from in-notebook specs.


In [15]:
def llm_text(system_prompt: str, user_prompt: str) -> str:
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return (response.choices[0].message.content or "").strip()


def preprocess_agent(state: ModelAgentState) -> dict[str, Any]:
    if state.raw_df is None or len(state.raw_df) == 0:
        raise ValueError("state.raw_df must contain the ingested dataset before preprocessing.")

    required_columns = TRAINING_COLUMNS + [TARGET_COLUMN]
    missing_columns = [column for column in required_columns if column not in state.raw_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df = state.raw_df[required_columns].copy()
    y = df[TARGET_COLUMN].copy()
    X = df[TRAINING_COLUMNS].copy()

    X[CATEGORICAL_TRAINING_COLUMNS] = X[CATEGORICAL_TRAINING_COLUMNS].fillna("Unknown")
    for column in NUMERIC_TRAINING_COLUMNS:
        X[column] = pd.to_numeric(X[column], errors="coerce")
        X[column] = X[column].fillna(X[column].median())

    X_processed = pd.get_dummies(X, columns=CATEGORICAL_TRAINING_COLUMNS, drop_first=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
    )

    return {
        "processed_df": pd.concat([X_processed, y.rename(TARGET_COLUMN)], axis=1),
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test,
        "feature_columns": X_processed.columns.tolist(),
        "messages": [
            f"[preprocess_agent] Prepared {len(df):,} rows into {X_processed.shape[1]} features using only the approved input columns."
        ],
    }


def random_forest_train_agent(state: ModelAgentState) -> dict[str, Any]:
    model = RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    )
    model.fit(state.X_train, state.y_train)
    return {
        "random_forest_model": model,
        "messages": ["[random_forest_train_agent] Random Forest trained successfully."],
    }


def xgboost_train_agent(state: ModelAgentState) -> dict[str, Any]:
    model = XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        tree_method="hist",
    )
    model.fit(state.X_train, state.y_train)
    return {
        "xgboost_model": model,
        "messages": ["[xgboost_train_agent] XGBoost trained successfully."],
    }


def evaluate_model(model: Any, X_test: pd.DataFrame, y_test: pd.Series, model_name: str, skill_name: str) -> tuple[dict[str, float], str]:
    y_pred = model.predict(X_test)
    metrics = {
        "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2": float(r2_score(y_test, y_pred)),
    }
    interpretation = llm_text(
        system_prompt=load_skill_body(skill_name),
        user_prompt=(
            f"Model: {model_name}\n"
            f"RMSE: ${metrics['rmse']:,.2f}\n"
            f"MAE: ${metrics['mae']:,.2f}\n"
            f"R2: {metrics['r2']:.4f}\n"
            "Explain what this means for a hiring team using the model to benchmark salary expectations. Keep it to 3 short sentences."
        ),
    )
    return metrics, interpretation


def random_forest_eval_agent(state: ModelAgentState) -> dict[str, Any]:
    metrics, interpretation = evaluate_model(
        state.random_forest_model,
        state.X_test,
        state.y_test,
        "Random Forest",
        "random_forest_eval",
    )
    return {
        "random_forest_evaluation_metrics": metrics,
        "messages": [
            f"[random_forest_eval_agent] RMSE=${metrics['rmse']:,.0f}, MAE=${metrics['mae']:,.0f}, R2={metrics['r2']:.3f}",
            f"[random_forest_eval_agent] {interpretation}",
        ],
    }


def xgboost_eval_agent(state: ModelAgentState) -> dict[str, Any]:
    metrics, interpretation = evaluate_model(
        state.xgboost_model,
        state.X_test,
        state.y_test,
        "XGBoost",
        "xgboost_eval",
    )
    return {
        "xgboost_evaluation_metrics": metrics,
        "messages": [
            f"[xgboost_eval_agent] RMSE=${metrics['rmse']:,.0f}, MAE=${metrics['mae']:,.0f}, R2={metrics['r2']:.3f}",
            f"[xgboost_eval_agent] {interpretation}",
        ],
    }


def model_selection_agent(state: ModelAgentState) -> dict[str, Any]:
    rf_metrics = state.random_forest_evaluation_metrics
    xgb_metrics = state.xgboost_evaluation_metrics

    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": load_skill_body("model_selection")},
            {
                "role": "user",
                "content": (
                    f"Random Forest metrics: {json.dumps(rf_metrics)}\n"
                    f"XGBoost metrics: {json.dumps(xgb_metrics)}\n"
                    "Return valid JSON with keys model_name and selection_reason."
                ),
            },
        ],
    )

    decision = json.loads(response.choices[0].message.content)
    selected_name = decision["model_name"]
    selection_reason = decision["selection_reason"]

    if selected_name == "XGBoost":
        selected_model = state.xgboost_model
    else:
        selected_name = "Random Forest"
        selected_model = state.random_forest_model

    return {
        "selected_model": selected_model,
        "selected_model_name": selected_name,
        "selection_reason": selection_reason,
        "messages": [f"[model_selection_agent] {selection_reason}"],
    }


In [16]:
graph_model = StateGraph(ModelAgentState)
graph_model.add_node("preprocess", preprocess_agent)
graph_model.add_node("random_forest_train", random_forest_train_agent)
graph_model.add_node("xgboost_train", xgboost_train_agent)
graph_model.add_node("random_forest_eval", random_forest_eval_agent)
graph_model.add_node("xgboost_eval", xgboost_eval_agent)
graph_model.add_node("model_selection", model_selection_agent)

graph_model.set_entry_point("preprocess")
graph_model.add_edge("preprocess", "random_forest_train")
graph_model.add_edge("preprocess", "xgboost_train")
graph_model.add_edge("random_forest_train", "random_forest_eval")
graph_model.add_edge("xgboost_train", "xgboost_eval")
graph_model.add_edge("random_forest_eval", "model_selection")
graph_model.add_edge("xgboost_eval", "model_selection")
graph_model.add_edge("model_selection", END)

app_model = graph_model.compile()
training_result = app_model.invoke(ModelAgentState(raw_df=job_df))

model_artifact = {
    "model": training_result["selected_model"],
    "model_name": training_result["selected_model_name"],
    "selection_reason": training_result["selection_reason"],
    "feature_columns": training_result["feature_columns"],
    "training_columns": TRAINING_COLUMNS,
    "categorical_columns": CATEGORICAL_TRAINING_COLUMNS,
    "numeric_columns": NUMERIC_TRAINING_COLUMNS,
}

with open(MODEL_PATH, "wb") as handle:
    pickle.dump(model_artifact, handle)

print(f"Saved selected model artifact to {MODEL_PATH}")
print("Selected model:", training_result["selected_model_name"])
print(training_result["selection_reason"])


Saved selected model artifact to models/selected_model.pkl
Selected model: XGBoost
XGBoost performs better because it has a lower RMSE of 11593.09 and a lower MAE of 9175.58, along with a higher R2 value of 0.93 compared to Random Forest's metrics.


## Inference Agent Pipeline - Gradio

In [16]:
# needle, add new pipeline here

def load_model_artifact() -> dict[str, Any]:
    if not os.path.exists(MODEL_PATH):
        raise FileNotFoundError(f"Model artifact not found at {MODEL_PATH}. Run the training pipeline first.")
    with open(MODEL_PATH, "rb") as handle:
        return pickle.load(handle)


def normalize_choice(value: Any, choices: list[str]) -> str:
    text = str(value).strip()
    if not text:
        raise ValueError("Every form field must have a value.")

    lowered_map = {choice.lower(): choice for choice in choices}
    return lowered_map.get(text.lower(), text)


def normalize_experience_years(value: Any) -> int:
    text = str(value).strip()
    if not text:
        raise ValueError("Year of experience is required.")

    years = int(float(text))
    return max(EXPERIENCE_MIN, min(EXPERIENCE_MAX, years))


def etl_gradio_input(raw_input: dict[str, Any]) -> dict[str, Any]:
    return {
        "job_role": normalize_choice(raw_input["job_role"], FEATURE_CHOICES["job_role"]),
        "experience_level": normalize_choice(raw_input["experience_level"], FEATURE_CHOICES["experience_level"]),
        "experience_years": normalize_experience_years(raw_input["experience_years"]),
        "country": normalize_choice(raw_input["country"], FEATURE_CHOICES["country"]),
        "education_required": normalize_choice(raw_input["education_required"], FEATURE_CHOICES["education_required"]),
        "industry": normalize_choice(raw_input["industry"], FEATURE_CHOICES["industry"]),
        "company_size": normalize_choice(raw_input["company_size"], FEATURE_CHOICES["company_size"]),
        "work_mode": normalize_choice(raw_input["work_mode"], FEATURE_CHOICES["work_mode"]),
    }


def prettify_feature_label(feature_name: str) -> str:
    for column in CATEGORICAL_TRAINING_COLUMNS:
        prefix = f"{column}_"
        if feature_name.startswith(prefix):
            return f"{column.replace('_', ' ').title()}: {feature_name[len(prefix):]}"
    return feature_name.replace("_", " ").title()


def explain_local_prediction(model: Any, encoded_row: pd.DataFrame, top_k: int = 8) -> tuple[Optional[float], list[dict[str, Any]]]:
    try:
        explanation = shap.TreeExplainer(model)(encoded_row)
    except Exception:
        return None, []

    shap_values = np.asarray(explanation.values[0], dtype=float)
    base_value = float(np.asarray(explanation.base_values).reshape(-1)[0])
    row_values = encoded_row.iloc[0]
    top_factors: list[dict[str, Any]] = []

    for feature_name, feature_value, shap_value in zip(encoded_row.columns, row_values.tolist(), shap_values.tolist()):
        if feature_name not in NUMERIC_TRAINING_COLUMNS and float(feature_value) == 0.0:
            continue
        if abs(shap_value) < 1e-6:
            continue

        top_factors.append(
            {
                "feature_name": feature_name,
                "label": prettify_feature_label(feature_name),
                "feature_value": feature_value,
                "shap_value": float(shap_value),
            }
        )

    top_factors.sort(key=lambda item: abs(item["shap_value"]), reverse=True)
    return base_value, top_factors[:top_k]


def format_shap_summary(top_factors: list[dict[str, Any]]) -> str:
    if not top_factors:
        return "- Local SHAP explanation was unavailable for this prediction."

    lines = []
    for item in top_factors:
        direction = "increased" if item["shap_value"] >= 0 else "reduced"
        feature_value = item["feature_value"]
        if isinstance(feature_value, float) and feature_value.is_integer():
            feature_value = int(feature_value)
        lines.append(
            f"- {item['label']} ({feature_value}) {direction} the estimate by about ${abs(item['shap_value']):,.0f}."
        )
    return "\n".join(lines)


def load_pickled_model_agent(state: PredictionAgentState) -> dict[str, Any]:
    artifact = load_model_artifact()
    return {
        "model_artifact": artifact,
        "messages": [f"[load_pickled_model_agent] Loaded {artifact['model_name']} artifact from disk."],
    }


def run_inference_agent(state: PredictionAgentState) -> dict[str, Any]:
    normalized_input = etl_gradio_input(state.raw_input)
    input_row = pd.DataFrame([normalized_input])

    encoded_row = pd.get_dummies(
        input_row,
        columns=CATEGORICAL_TRAINING_COLUMNS,
        drop_first=False,
    )
    encoded_row = encoded_row.reindex(
        columns=state.model_artifact["feature_columns"],
        fill_value=0,
    )

    prediction = float(state.model_artifact["model"].predict(encoded_row)[0])
    prediction_low = prediction * 0.9
    prediction_high = prediction * 1.1
    shap_base_value, shap_top_factors = explain_local_prediction(state.model_artifact["model"], encoded_row)
    shap_summary = format_shap_summary(shap_top_factors)

    return {
        "normalized_input": normalized_input,
        "prepared_features": encoded_row,
        "prediction": prediction,
        "prediction_low": prediction_low,
        "prediction_high": prediction_high,
        "shap_base_value": shap_base_value,
        "shap_top_factors": shap_top_factors,
        "shap_summary": shap_summary,
        "messages": [
            f"[run_inference_agent] ETL completed for {normalized_input['job_role']} in {normalized_input['country']}.",
            f"[run_inference_agent] Predicted ${prediction:,.0f} with a suggested range of ${prediction_low:,.0f} to ${prediction_high:,.0f}.",
            "[run_inference_agent] Local SHAP explanation computed for the submitted profile." if shap_top_factors else "[run_inference_agent] Local SHAP explanation was unavailable for this prediction.",
        ],
    }


def generate_recommendation_agent(state: PredictionAgentState) -> dict[str, Any]:
    profile = state.normalized_input or state.raw_input
    interpretation = llm_text(
        system_prompt=load_skill_body("generate_recommendation"),
        user_prompt=(
            "Use the model output below to write a concise business interpretation for a hiring manager. "
            "Keep it to 3 short bullet points in Markdown, do not invent a different salary number, and use the local SHAP drivers to explain what most influenced the estimate.\n\n"
            f"Model: {state.model_artifact['model_name']}\n"
            f"Job role: {profile['job_role']}\n"
            f"Experience level: {profile['experience_level']}\n"
            f"Years of experience: {profile['experience_years']}\n"
            f"Country: {profile['country']}\n"
            f"Education: {profile['education_required']}\n"
            f"Industry: {profile['industry']}\n"
            f"Company size: {profile['company_size']}\n"
            f"Work mode: {profile['work_mode']}\n"
            f"Predicted salary: ${state.prediction:,.0f}\n"
            f"Suggested range: ${state.prediction_low:,.0f} to ${state.prediction_high:,.0f}\n\n"
            f"Top local SHAP drivers:\n{state.shap_summary}\n"
        ),
    )
    return {
        "business_interpretation": interpretation,
        "messages": ["[generate_recommendation_agent] Business interpretation generated for the frontend using the local SHAP drivers."],
    }


graph_prediction = StateGraph(PredictionAgentState)
graph_prediction.add_node("load_pickled_model", load_pickled_model_agent)
graph_prediction.add_node("run_inference", run_inference_agent)
graph_prediction.add_node("generate_recommendation", generate_recommendation_agent)

graph_prediction.set_entry_point("load_pickled_model")
graph_prediction.add_edge("load_pickled_model", "run_inference")
graph_prediction.add_edge("run_inference", "generate_recommendation")
graph_prediction.add_edge("generate_recommendation", END)

app_prediction = graph_prediction.compile()


In [17]:
def editable_dropdown(label: str, choices: list[str], value: str) -> gr.Dropdown:
    return gr.Dropdown(
        choices=choices,
        label=label,
        value=value,
        allow_custom_value=True,
        filterable=True,
        interactive=True,
    )


def build_local_shap_plot(
    top_factors: list[dict[str, Any]],
    prediction: float,
    base_value: Optional[float],
):
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.65 * max(len(top_factors), 1) + 1.2)))

    if not top_factors:
        ax.axis("off")
        ax.text(
            0.5,
            0.5,
            "Local SHAP explanation unavailable for this prediction.",
            ha="center",
            va="center",
            fontsize=12,
        )
        fig.tight_layout()
        return fig

    plot_items = list(reversed(top_factors))
    labels = [item["label"] for item in plot_items]
    values = [item["shap_value"] for item in plot_items]
    colors = ["#1f9d55" if value >= 0 else "#d95f02" for value in values]
    max_abs_value = max(abs(value) for value in values)
    label_offset = max_abs_value * 0.04 if max_abs_value else 1.0

    ax.barh(labels, values, color=colors)
    ax.axvline(0, color="#4b5563", linewidth=1)
    ax.set_xlabel("Contribution to salary estimate (USD)")
    if base_value is not None:
        ax.set_title(f"Local SHAP impact: base ${base_value:,.0f} -> prediction ${prediction:,.0f}")
    else:
        ax.set_title(f"Local SHAP impact on prediction ${prediction:,.0f}")

    for index, value in enumerate(values):
        x_position = value + label_offset if value >= 0 else value - label_offset
        alignment = "left" if value >= 0 else "right"
        ax.text(x_position, index, f"{value:+,.0f}", va="center", ha=alignment, fontsize=10)

    fig.tight_layout()
    return fig


def predict_salary(
    job_role: str,
    experience_level: str,
    experience_years: str,
    country: str,
    education_required: str,
    industry: str,
    company_size: str,
    work_mode: str,
) -> tuple[str, Any, str, str]:
    result = app_prediction.invoke(
        PredictionAgentState(
            raw_input={
                "job_role": job_role,
                "experience_level": experience_level,
                "experience_years": experience_years,
                "country": country,
                "education_required": education_required,
                "industry": industry,
                "company_size": company_size,
                "work_mode": work_mode,
            }
        )
    )

    normalized_input = result["normalized_input"]
    salary_summary = (
        f"### Salary Estimate\n"
        f"**${result['prediction']:,.0f}**\n\n"
        f"Suggested range: **${result['prediction_low']:,.0f} to ${result['prediction_high']:,.0f}**\n\n"
        f"Selected model: **{result['model_artifact']['model_name']}**\n"
        f"Profile used: **{normalized_input['job_role']} | {normalized_input['experience_level']} | {normalized_input['experience_years']} yrs | {normalized_input['country']}**"
    )
    pipeline_trace = "### Agent Pipeline\n" + "\n".join(f"- {message}" for message in result["messages"])
    shap_plot = build_local_shap_plot(
        result.get("shap_top_factors") or [],
        result["prediction"],
        result.get("shap_base_value"),
    )
    return salary_summary, shap_plot, result["business_interpretation"], pipeline_trace


with gr.Blocks() as demo:
    gr.Markdown("# AI Talent Salary Prediction System")
    gr.Markdown("Editable dropdowns feed the agent pipeline for ETL, model inference, local SHAP explanation, and LLM interpretation.")

    with gr.Row():
        with gr.Column():
            job_role = editable_dropdown(
                label="Job Role",
                choices=FEATURE_CHOICES["job_role"],
                value=FEATURE_CHOICES["job_role"][0],
            )
            experience_level = editable_dropdown(
                label="Experience Level",
                choices=FEATURE_CHOICES["experience_level"],
                value=FEATURE_CHOICES["experience_level"][0],
            )
            experience_years = editable_dropdown(
                label="Year of Exp",
                choices=[str(year) for year in range(EXPERIENCE_MIN, EXPERIENCE_MAX + 1)],
                value=str(min(5, EXPERIENCE_MAX)),
            )
            country = editable_dropdown(
                label="Country",
                choices=FEATURE_CHOICES["country"],
                value=FEATURE_CHOICES["country"][0],
            )

        with gr.Column():
            education_required = editable_dropdown(
                label="Education",
                choices=FEATURE_CHOICES["education_required"],
                value=FEATURE_CHOICES["education_required"][0],
            )
            industry = editable_dropdown(
                label="Industry",
                choices=FEATURE_CHOICES["industry"],
                value=FEATURE_CHOICES["industry"][0],
            )
            company_size = editable_dropdown(
                label="Company Size",
                choices=FEATURE_CHOICES["company_size"],
                value=FEATURE_CHOICES["company_size"][0],
            )
            work_mode = editable_dropdown(
                label="Work Mode",
                choices=FEATURE_CHOICES["work_mode"],
                value=FEATURE_CHOICES["work_mode"][0],
            )

    predict_btn = gr.Button("Predict Salary", variant="primary", size="lg")

    salary_output = gr.Markdown(label="Salary Estimate")
    shap_output = gr.Plot(label="Local SHAP Feature Impact")
    interpretation_output = gr.Markdown(label="Business Interpretation")
    pipeline_output = gr.Markdown(label="Pipeline Trace")

    predict_btn.click(
        fn=predict_salary,
        inputs=[
            job_role,
            experience_level,
            experience_years,
            country,
            education_required,
            industry,
            company_size,
            work_mode,
        ],
        outputs=[salary_output, shap_output, interpretation_output, pipeline_output],
    )

demo.launch(share=True, theme=gr.themes.Soft())


* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 08:12:45 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T08:12:45Z is after 2026-04-01T07:01:35Z
